<a href="https://colab.research.google.com/github/rokeya160210/Skin-Cancer-2-papers-comparison/blob/main/04_efficiency_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Model size, FLOPs, peak GPU memory and inference latency (#18)

For InceptionV3, DenseNet121, ConvNeXt-Large, H2 (InceptionV3 + DenseNet121), H3 (all three), and optionally EViT-Dens169:

* **total and trainable parameters**, fp32 parameter memory, size of the saved `.keras` checkpoint;
* **FLOPs per image** from the TensorFlow profiler on the inference graph (batch 1). The profiler counts a multiply-add as 2 FLOPs; GMACs = GFLOPs / 2 is also given;
* **peak GPU memory** for inference (batch 1 and batch 32) and for one training step at the batch size actually used (16; EViT 8);
* **latency**: median and IQR ms/image over 100 timed calls after 20 warm-up calls. Batch 1 is the per-image interactive latency; batch 32 is throughput per image. For H2/H3 the members run sequentially inside one `tf.function` followed by the weighted sum, which is how the ensemble would be deployed.

Every measurement runs in **its own Python process**, so TensorFlow's allocator cache cannot inflate later numbers. Weights are randomly initialised: none of these quantities depends on weight values.

**Report the GPU model, driver, TF version and CUDA version printed in the first cell alongside the table.** Latency is hardware-specific. Run this on the GPU you trained on (your ensemble runs used an A100-SXM4-40GB; EViT used an L4).

In [1]:
# ---- connect Google Drive FIRST (before any folder is created) ------------------------------
import os, time
def _drive_is_mounted():
    try:
        return any(" /content/drive" in l for l in open("/proc/mounts"))
    except Exception:
        return os.path.ismount("/content/drive")
def mount_google_drive():
    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Colab -> Google Drive not mounted (local paths are used).")
        return False
    if not _drive_is_mounted() and os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        # A plain local folder (e.g. created by an earlier run before Drive was connected) would hide
        # Google Drive. It is NOT your Drive: move it aside (nothing is deleted).
        stray = f"/content/drive_local_copy_{int(time.time())}"
        os.rename("/content/drive", stray)
        print(f"Moved a local folder that was blocking Google Drive to {stray}")
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError("Google Drive could not be connected. Runtime -> Disconnect and delete runtime, run this "
                           "cell alone, choose the account that owns your Drive and tick 'Select all'. "
                           f"(Colab said: {e})")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive is not visible at /content/drive/MyDrive.")
    print("Google Drive connected.")
    return True
IN_COLAB = mount_google_drive()
import sys, json, subprocess, platform
import pandas as pd
INCLUDE_EVIT = True
OUT_DIR = "/content/drive/MyDrive/skin_ablation_v2/profiling" if IN_COLAB else os.path.abspath("profiling")
os.makedirs(OUT_DIR, exist_ok=True)
env = {"python": platform.python_version()}
try:
    env["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                                capture_output=True, text=True).stdout.strip()
except Exception:
    env["gpu"] = "none (CPU)"
env["tensorflow"] = subprocess.run([sys.executable, "-c", "import tensorflow as tf; print(tf.__version__, tf.sysconfig.get_build_info().get('cuda_version'))"],
                                   capture_output=True, text=True).stdout.strip()
print(json.dumps(env, indent=2))

Mounted at /content/drive
Google Drive connected.
{
  "python": "3.13.15",
  "gpu": "NVIDIA A100-SXM4-40GB, 580.82.07, 40960 MiB",
  "tensorflow": "2.21.0 12.5.1"
}


In [2]:
WORKER_SRC = r'''
"""profile_worker.py -- one measurement per process, so peak-memory numbers are not contaminated."""
import os, sys, json, time, tempfile
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3, DenseNet121, ConvNeXtLarge
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
spec = json.loads(sys.argv[1])
NUM_CLASSES, IMG = 6, (224, 224)
W = None   # random init: parameter counts, FLOPs, memory and latency do not depend on weight values

# ---- builders: identical layer graphs to notebook 02 (augmentation layers are inactive at inference) ----
def pre01(inp):
    x = layers.RandomFlip("horizontal")(inp); x = layers.Rescaling(1 / 255)(x); return layers.GaussianNoise(10 / 255)(x)
def inception():
    b = InceptionV3(include_top=False, weights=W, input_shape=IMG + (3,)); i = layers.Input(IMG + (3,))
    x = layers.GlobalAveragePooling2D()(b(pre01(i))); x = layers.Dense(512, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x))
def densenet():
    b = DenseNet121(include_top=False, weights=W, input_shape=IMG + (3,)); i = layers.Input(IMG + (3,))
    x = layers.Conv2D(512, 3, padding="same", activation="relu")(b(pre01(i))); x = layers.GlobalMaxPooling2D()(x)
    x = layers.Dense(1024, activation="relu")(x); x = layers.Dense(512, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x))
def convnext():
    b = ConvNeXtLarge(include_top=False, weights=W, input_shape=IMG + (3,)); b.trainable = False; i = layers.Input(IMG + (3,))
    x = layers.Rescaling(255.0)(pre01(i)); x = layers.GlobalAveragePooling2D()(b(x)); x = layers.Dense(512, activation="relu")(x)
    return models.Model(i, layers.Dense(NUM_CLASSES, activation="softmax")(x))

def evit():   # compact copy of the EViT-Dens169 re-implementation (same hyper-parameters as the notebook)
    H, Wd, P, D, NH, NB, M, FD, DR = 450, 600, 30, 128, 8, 6, 256, 256, 0.3
    def dl(x, g):
        c = layers.BatchNormalization()(x); c = layers.Activation("relu")(c); c = layers.Conv2D(4 * g, 1, padding="same", use_bias=False)(c)
        c = layers.BatchNormalization()(c); c = layers.Activation("relu")(c); c = layers.Conv2D(g, 3, padding="same", use_bias=False)(c)
        return layers.Concatenate()([x, c])
    def tr(x):
        f = max(int(x.shape[-1] * 0.5), 1); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
        return layers.AveragePooling2D(2, 2)(layers.Conv2D(f, 1, padding="same", use_bias=False)(x))
    inp = layers.Input((H, Wd, 3))
    x = layers.Conv2D(32, 3, strides=2, padding="same", use_bias=False)(inp); x = layers.Activation("relu")(layers.BatchNormalization()(x))
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    for n in (6, 12, 16):
        for _ in range(n): x = dl(x, 32)
        x = tr(x)
    fd = layers.GlobalAveragePooling2D()(layers.Activation("relu")(x))
    pe = layers.Conv2D(D, P, strides=P)(inp)
    pe = layers.Conv2D(D, 1, padding="same", activation="relu")(layers.Concatenate()([
        layers.Conv2D(64, 3, padding="same", activation="relu")(pe), layers.Conv2D(128, 5, padding="same", activation="relu")(pe),
        layers.Conv2D(256, 7, padding="same", activation="relu")(pe)]))
    n = (H // P) * (Wd // P); tok = layers.Reshape((n, D))(pe)
    pos = np.arange(n)[:, None]; ii = np.arange(D)[None, :]; a = pos / np.power(10000, (2 * (ii // 2)) / np.float32(D))
    a[:, 0::2] = np.sin(a[:, 0::2]); a[:, 1::2] = np.cos(a[:, 1::2]); tok = tok + tf.constant(a[None], tf.float32)
    for _ in range(NB):
        h = layers.LayerNormalization(epsilon=1e-6)(tok); tok = layers.Add()([tok, layers.MultiHeadAttention(NH, D // NH, dropout=DR)(h, h)])
        h = layers.LayerNormalization(epsilon=1e-6)(tok); m = layers.Dense(D)(layers.Dropout(DR)(layers.Dense(M, activation="gelu")(h)))
        tok = layers.Add()([tok, m])
    fe = layers.Flatten()(tok)
    z = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([layers.Dense(FD)(fe), layers.Dense(FD)(fd)]))
    z = layers.Dropout(DR)(layers.Dropout(DR)(layers.Flatten()(z))); z = layers.Dense(256, activation="relu")(z)
    z = layers.Dense(128, activation="relu")(layers.Dropout(DR)(z))
    return models.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(z))

BUILD = {"InceptionV3": inception, "DenseNet121": densenet, "ConvNeXt-Large": convnext, "EViT-Dens169": evit}
MEMBERS = {"H2 (Inc+Den)": ["InceptionV3", "DenseNet121"], "H3 (Inc+Den+Cnx)": ["InceptionV3", "DenseNet121", "ConvNeXt-Large"]}
WEIGHTS = {"H2 (Inc+Den)": [0.4545, 0.5455], "H3 (Inc+Den+Cnx)": [0.25, 0.30, 0.45]}
name, task = spec["name"], spec["task"]
members = MEMBERS.get(name, [name])
shape = (450, 600, 3) if name == "EViT-Dens169" else IMG + (3,)
ms = [BUILD[m]() for m in members]
gpu = bool(tf.config.list_physical_devices("GPU"))
def reset():
    if gpu: tf.config.experimental.reset_memory_stats("GPU:0")
def peak():
    return round(tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2**20, 1) if gpu else None
out = {"name": name, "task": task}

if task == "size":
    out["total_params"] = int(sum(m.count_params() for m in ms))
    out["trainable_params"] = int(sum(sum(int(np.prod(w.shape)) for w in m.trainable_weights) for m in ms))
    out["param_memory_MB_fp32"] = round(out["total_params"] * 4 / 2**20, 1)
    fl = 0
    for m in ms:   # profile the inference graph (variables are not frozen in, which keeps RAM low for ConvNeXt-L)
        cf = tf.function(lambda x, m=m: m(x, training=False)).get_concrete_function(tf.TensorSpec((1,) + shape, tf.float32))
        prof = tf.compat.v1.profiler.profile(graph=cf.graph, run_meta=tf.compat.v1.RunMetadata(), cmd="op",
                                             options=tf.compat.v1.profiler.ProfileOptionBuilder.float_operation())
        fl += prof.total_float_ops
    size = 0
    for m in ms:
        f = tempfile.NamedTemporaryFile(suffix=".keras", delete=False).name; m.save(f); size += os.path.getsize(f); os.remove(f)
    out["checkpoint_MB"] = round(size / 2**20, 1)
    out["GFLOPs_per_image"] = round(fl / 1e9, 3); out["GMACs_per_image"] = round(fl / 2e9, 3)

elif task == "latency":
    bs = spec["batch"]; x = tf.random.uniform((bs,) + shape, 0, 255)
    w = WEIGHTS.get(name, [1.0])
    @tf.function(jit_compile=False)
    def run(x):
        return tf.add_n([wi * m(x, training=False) for wi, m in zip(w, ms)])
    for _ in range(spec.get("warmup", 20)): run(x).numpy()
    reset(); t = []
    for _ in range(spec.get("iters", 100)):
        t0 = time.perf_counter(); run(x).numpy(); t.append(time.perf_counter() - t0)
    t = np.array(t) * 1000 / bs
    out.update({"batch": bs, "ms_per_image_median": round(float(np.median(t)), 3),
                "ms_per_image_IQR": [round(float(np.percentile(t, 25)), 3), round(float(np.percentile(t, 75)), 3)],
                "peak_GPU_MB_inference": peak()})

elif task == "train_step":
    # peak memory of one optimisation step at the training batch size; members are trained one at a time,
    # so an ensemble's training peak is the maximum over its members (each measured in its own process).
    bs = spec["batch"]; m = ms[0]
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy")
    x = tf.random.uniform((bs,) + shape, 0, 255); yy = tf.random.uniform((bs,), 0, NUM_CLASSES, tf.int32)
    if name == "EViT-Dens169": yy = tf.one_hot(yy, NUM_CLASSES); m.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="categorical_crossentropy")
    m.train_on_batch(x, yy); reset()
    t0 = time.perf_counter()
    for _ in range(spec.get("iters", 10)): m.train_on_batch(x, yy)
    out.update({"batch": bs, "peak_GPU_MB_train": peak(), "s_per_train_step": round((time.perf_counter() - t0) / spec.get("iters", 10), 4)})
print("RESULT " + json.dumps(out))
'''
WORKER = os.path.join(OUT_DIR, 'profile_worker.py')
open(WORKER, 'w').write(WORKER_SRC)
print('worker written to', WORKER)

worker written to /content/drive/MyDrive/skin_ablation_v2/profiling/profile_worker.py


In [3]:
def run(spec, timeout=3600):
    r = subprocess.run([sys.executable, WORKER, json.dumps(spec)], capture_output=True, text=True, timeout=timeout)
    line = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
    if not line:
        print("FAILED", spec, r.stderr[-2000:]); return None
    res = json.loads(line[0][7:]); print(res); return res

MODELS = ["InceptionV3", "DenseNet121", "ConvNeXt-Large", "H2 (Inc+Den)", "H3 (Inc+Den+Cnx)"] + (["EViT-Dens169"] if INCLUDE_EVIT else [])
R = {m: {} for m in MODELS}
for m in MODELS:
    R[m].update(run({"name": m, "task": "size"}) or {})
    for bs in [1, 32]:
        r = run({"name": m, "task": "latency", "batch": bs, "warmup": 20, "iters": 100})
        if r:
            R[m][f"ms/img (batch {bs}), median"] = r["ms_per_image_median"]
            R[m][f"ms/img (batch {bs}), IQR"] = r["ms_per_image_IQR"]
            R[m][f"peak GPU MB inference (batch {bs})"] = r["peak_GPU_MB_inference"]
for m, bs in [("InceptionV3", 16), ("DenseNet121", 16), ("ConvNeXt-Large", 16)] + ([("EViT-Dens169", 8)] if INCLUDE_EVIT else []):
    r = run({"name": m, "task": "train_step", "batch": bs, "iters": 10})
    if r:
        R[m]["peak GPU MB train step"] = r["peak_GPU_MB_train"]; R[m]["train batch"] = bs; R[m]["s per train step"] = r["s_per_train_step"]
for ens, mem in [("H2 (Inc+Den)", ["InceptionV3", "DenseNet121"]), ("H3 (Inc+Den+Cnx)", ["InceptionV3", "DenseNet121", "ConvNeXt-Large"])]:
    vals = [R[x].get("peak GPU MB train step") for x in mem]
    R[ens]["peak GPU MB train step"] = max(v for v in vals if v is not None) if any(vals) else None
    R[ens]["train batch"] = 16

{'name': 'InceptionV3', 'task': 'size', 'total_params': 22854950, 'trainable_params': 22820518, 'param_memory_MB_fp32': 87.2, 'checkpoint_MB': 88.2, 'GFLOPs_per_image': 5.696, 'GMACs_per_image': 2.848}
{'name': 'InceptionV3', 'task': 'latency', 'batch': 1, 'ms_per_image_median': 11.078, 'ms_per_image_IQR': [10.815, 11.498], 'peak_GPU_MB_inference': 116.0}
{'name': 'InceptionV3', 'task': 'latency', 'batch': 32, 'ms_per_image_median': 0.427, 'ms_per_image_IQR': [0.423, 0.434], 'peak_GPU_MB_inference': 344.0}
{'name': 'DenseNet121', 'task': 'size', 'total_params': 12809798, 'trainable_params': 12726150, 'param_memory_MB_fp32': 48.9, 'checkpoint_MB': 50.3, 'GFLOPs_per_image': 6.165, 'GMACs_per_image': 3.083}
{'name': 'DenseNet121', 'task': 'latency', 'batch': 1, 'ms_per_image_median': 16.308, 'ms_per_image_IQR': [16.079, 16.56], 'peak_GPU_MB_inference': 110.7}
{'name': 'DenseNet121', 'task': 'latency', 'batch': 32, 'ms_per_image_median': 0.794, 'ms_per_image_IQR': [0.789, 0.8], 'peak_GPU_M

In [4]:
T = pd.DataFrame(R).T
cols = ["total_params", "trainable_params", "checkpoint_MB", "GFLOPs_per_image", "GMACs_per_image",
        "peak GPU MB inference (batch 1)", "peak GPU MB inference (batch 32)", "peak GPU MB train step", "train batch",
        "ms/img (batch 1), median", "ms/img (batch 1), IQR", "ms/img (batch 32), median"]
T = T[[c for c in cols if c in T.columns]]
T.to_csv(os.path.join(OUT_DIR, "T_efficiency.csv"))
print(T.to_string())
json.dump(env, open(os.path.join(OUT_DIR, "profiling_environment.json"), "w"), indent=2)

def M(x): return f"{x/1e6:.2f}M" if pd.notna(x) else "-"
tex = T.copy()
for c in ["total_params", "trainable_params"]:
    if c in tex: tex[c] = tex[c].map(M)
keep = [c for c in ["total_params", "trainable_params", "GFLOPs_per_image", "peak GPU MB inference (batch 1)",
                    "peak GPU MB train step", "ms/img (batch 1), median", "ms/img (batch 32), median"] if c in tex]
L = [r"\begin{table}[t]", r"\centering",
     r"\caption{Model size, compute and latency (" + env.get("gpu", "") + r", TensorFlow " + env.get("tensorflow", "") +
     r"). FLOPs count a multiply-add as 2; H2/H3 inference runs members sequentially plus the weighted sum; "
     r"training peak memory of an ensemble is the maximum over its separately trained members.}",
     r"\label{tab:efficiency}", r"\small", r"\begin{tabular}{l" + "c" * len(keep) + "}", r"\toprule",
     "Model & Params & Trainable & GFLOPs/img & Peak MB (inf., b=1) & Peak MB (train) & ms/img (b=1) & ms/img (b=32) \\\\", r"\midrule"]
for n, r in tex[keep].iterrows():
    L.append(n + " & " + " & ".join(str(v) for v in r.values) + r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(os.path.join(OUT_DIR, "T_efficiency.tex"), "w").write("\n".join(L)); print("\n".join(L))

                 total_params trainable_params checkpoint_MB GFLOPs_per_image GMACs_per_image peak GPU MB inference (batch 1) peak GPU MB inference (batch 32) peak GPU MB train step train batch ms/img (batch 1), median ms/img (batch 1), IQR ms/img (batch 32), median
InceptionV3          22854950         22820518          88.2            5.696           2.848                           116.0                            344.0                 1018.4          16                   11.078      [10.815, 11.498]                     0.427
DenseNet121          12809798         12726150          50.3            6.165           3.083                           110.7                            344.6                 1607.9          16                   16.308       [16.079, 16.56]                     0.794
ConvNeXt-Large      197020358           790022         752.5           68.284          34.142                           818.1                           1460.1                 1110.4          16      